In [1]:
# | default_exp wuilt.sync

# Wuilt Sync
> Fetch products from the Wuilt API and upsert them into the local database.

In [2]:
# | export
from seootter.wuilt.client import WuiltClient
from seootter.models import add_or_update_wuilt_store, upsert_wuilt_product, upsert_wuilt_page, WuiltProduct, WuiltPage, WuiltStore
from seootter.sqlite_db import get_session
from sqlmodel import select
from datetime import datetime


In [3]:
# | export
def sync_wuilt_products(
    api_key: str,
    store_id: str,
    store_name: str = "",
    locale: str = "ar",
    db_url: str | None = None,
) -> dict:
    "Fetch all products from Wuilt API and upsert into local DB. Returns sync stats."
    client = WuiltClient(api_key=api_key, store_id=store_id, locale=locale)

    api_products = client.get_products()

    stats = {"total": len(api_products), "created": 0, "updated": 0}

    with get_session(db_url) as session:
        store = add_or_update_wuilt_store(
            session=session,
            store_id=store_id,
            name=store_name or store_id,
            api_key=api_key,
            locale=locale,
        )

        for p in api_products:
            existed = session.exec(
                select(WuiltProduct).where(
                    WuiltProduct.wuilt_store_id == store.id,
                    WuiltProduct.wuilt_product_id == p["id"],
                )
            ).first()

            variants = p.get("variants", {}) or {}
            nodes = variants.get("nodes", []) if isinstance(variants, dict) else []
            price = None
            if nodes:
                first_price = nodes[0].get("price", {}) or {}
                price = first_price.get("amount")

            images = p.get("images", []) or []
            image_urls = [img.get("src", "") for img in images if img.get("src")]
            images_alt = [img.get("altText", "") or "" for img in images]

            seo = p.get("seo") or {}

            upsert_wuilt_product(
                session=session,
                wuilt_store_id=store.id,
                wuilt_product_id=p["id"],
                title=p.get("title", ""),
                handle=p.get("handle", ""),
                seo_title=seo.get("title"),
                seo_description=seo.get("description"),
                description_html=p.get("descriptionHtml"),
                short_description=p.get("shortDescription"),
                price=price,
                image_urls=image_urls,
                images_alt=images_alt,
                raw_data=p,
            )

            if existed:
                stats["updated"] += 1
            else:
                stats["created"] += 1

        session.commit()

    return stats

In [4]:
# | export
def sync_wuilt_collections(
    api_key: str,
    store_id: str,
    locale: str = "ar",
    db_url: str | None = None,
) -> dict:
    "Fetch all collections from Wuilt API and return them. (Collection storage TBD.)"
    client = WuiltClient(api_key=api_key, store_id=store_id, locale=locale)
    collections = client.get_collections()
    return {"total": len(collections), "collections": collections}

In [5]:
# | export
def sync_wuilt_store(
    api_key: str,
    store_id: str,
    store_name: str = "",
    locale: str = "ar",
    db_url: str | None = None,
) -> dict:
    "Fetch store info from Wuilt API and update local store record."
    client = WuiltClient(api_key=api_key, store_id=store_id, locale=locale)
    api_store = client.get_store()
    stats = {"seo_title": None, "seo_description": None, "robots_txt": ""}
    seo = api_store.get("seo") or {}
    stats["seo_title"] = seo.get("title")
    stats["seo_description"] = seo.get("description")
    robots = api_store.get("robotsTxt") or {}
    stats["robots_txt"] = robots.get("content") or ""
    return stats


In [6]:
# | export
def sync_wuilt_pages(
    api_key: str,
    store_id: str,
    store_name: str = "",
    locale: str = "ar",
    db_url: str | None = None,
) -> dict:
    "Fetch all pages from Wuilt API and upsert into local DB."
    client = WuiltClient(api_key=api_key, store_id=store_id, locale=locale)
    api_pages = client.get_pages()
    stats = {"total": len(api_pages), "created": 0, "updated": 0}
    with get_session(db_url) as session:
        store = add_or_update_wuilt_store(
            session=session,
            store_id=store_id,
            name=store_name or store_id,
            api_key=api_key,
            locale=locale,
        )
        for p in api_pages:
            existed = session.exec(
                select(WuiltPage).where(
                    WuiltPage.wuilt_store_id == store.id,
                    WuiltPage.wuilt_page_id == p["id"],
                )
            ).first()
            seo = p.get("seo") or {}
            upsert_wuilt_page(
                session=session,
                wuilt_store_id=store.id,
                wuilt_page_id=p["id"],
                name=p.get("name", ""),
                handle=p.get("handle", ""),
                page_type=p.get("pageType", "CUSTOM"),
                status=p.get("status", "DRAFT"),
                locale=p.get("locale", locale),
                seo_title=seo.get("title"),
                seo_description=seo.get("description"),
                raw_data=p,
            )
            if existed:
                stats["updated"] += 1
            else:
                stats["created"] += 1
        session.commit()
    return stats


In [7]:
# | export
def full_wuilt_sync(
    api_key: str,
    store_id: str,
    store_name: str = "",
    locale: str = "ar",
    db_url: str | None = None,
) -> dict:
    "Sync store info, products, pages, and collections from Wuilt API into local DB."
    store = sync_wuilt_store(
        api_key=api_key, store_id=store_id,
        store_name=store_name, locale=locale, db_url=db_url,
    )
    products = sync_wuilt_products(
        api_key=api_key, store_id=store_id,
        store_name=store_name, locale=locale, db_url=db_url,
    )
    pages = sync_wuilt_pages(
        api_key=api_key, store_id=store_id,
        store_name=store_name, locale=locale, db_url=db_url,
    )
    collections = sync_wuilt_collections(
        api_key=api_key, store_id=store_id,
        locale=locale, db_url=db_url,
    )
    return {
        "store": store,
        "products": products,
        "pages": pages,
        "collections": collections,
    }


In [9]:
# | hide
# | eval: false
from dotenv import load_dotenv
import os

load_dotenv()

result = full_wuilt_sync(
    api_key=os.getenv("WUILT_API_KEY"),
    store_id=os.getenv("WUILT_STORE_ID"),
    store_name="HelixGain",
)
print(f"Products: {result['products']}")
print(f"Collections: {result['collections']}")
print(f"Pages: {result['pages']}")
print(
    f"Store SEO: {result['store']['seo_title']} - {result['store']['seo_description']}"
)

Products: {'total': 52, 'created': 0, 'updated': 52}
Collections: {'total': 0, 'collections': []}
Pages: {'total': 0, 'created': 0, 'updated': 0}
Store SEO: None - None
